# 1 Data Cleaning

Flow mới:
1. **Input**: `data/raw_dirty/jobs_in_data_dirty.csv` (sinh từ raw bằng `src.data.dirtify`).
2. **Cleaning steps** (đối ứng với từng loại noise đã inject):
   - Standardize column names về snake_case.
   - Strip / normalize case ở cột categorical.
   - Sửa giá trị enum sai chính tả (mapping).
   - Parse `salary` về numeric (loại `$`, `,`, khoảng trắng).
   - Handle missing values.
   - Remove duplicates.
   - Detect & xử lý outlier `salary_in_usd`.
   - Final dtype conversion.
3. **Output**: upload thẳng lên **MinIO** bucket `${BUCKET_NAME}` ở 2 format (CSV cho Power BI, Parquet cho downstream). Không ghi `data/processed/` local.

In [1]:
# Setup: project root + imports
import os
import re
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

# Clear cached `src.*` modules so re-runs pick up edits
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

import yaml
import numpy as np
import pandas as pd

from src.data.dirtify import make_dirty_dataset
from src.data.storage import upload_dataframe, get_bucket_name

In [2]:
# Load config + ensure dirty dataset exists
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

raw_path = Path(config["paths"]["raw_data"]) / config["data"]["raw_filename"]
dirty_path = Path(config["paths"]["dirty_data"]) / config["data"]["dirty_filename"]

if not dirty_path.exists():
    make_dirty_dataset(raw_path=raw_path, dirty_path=dirty_path, seed=config["data"]["dirty_seed"])
else:
    print(f"Dirty dataset already exists: {dirty_path}")

df = pd.read_csv(dirty_path)
print(f"Loaded dirty data: {df.shape[0]} rows x {df.shape[1]} cols")
df.head()

Dirty dataset already exists: data\raw_dirty\jobs_in_data_dirty.csv
Loaded dirty data: 9495 rows x 12 cols


,work_year,job_title,job_category,salary_currency,salary,Salary In USD,employee_residence,experience_level,employment_type,work_setting,company_location,CompanySize
0,2023,Data DevOps Engineer,Data Engineering,EUR,88000,95012,Germany,Mid level,Full-time,Hybrid,Germany,L
1,2023,Data Architect,Data Architecture and Modeling,USD,186000,186000,United States,Senior,Full-time,In-person,United States,M
2,2023,Data Architect,Data Architecture and Modeling,USD,81800,81800,United States,senior,Full-time,In-person,United States,M
3,2023,Data Scientist,Data Science and Research,USD,"212,000",212000,United States,Senior,Full-time,In-person,United States,M
4,2023,Data Scientist,Data Science and Research,USD,93300,93300,United States,Senior,Full-time,In-person,United States,M


In [3]:
# Step 1: Standardize column names -> snake_case
def to_snake_case(name: str) -> str:
    s = name.strip()
    s = re.sub(r"[\s\-]+", "_", s)
    # camelCase + acronym boundaries
    s = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", s)
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)
    s = re.sub(r"_+", "_", s)
    return s.lower()

df.columns = [to_snake_case(c) for c in df.columns]
print(df.columns.tolist())

['work_year', 'job_title', 'job_category', 'salary_currency', 'salary', 'salary_in_usd', 'employee_residence', 'experience_level', 'employment_type', 'work_setting', 'company_location', 'company_size']


In [4]:
# Step 2: Strip whitespace + normalize case in categorical columns
CATEGORICAL_COLS = [
    "job_title", "job_category", "salary_currency",
    "employee_residence", "experience_level", "employment_type",
    "work_setting", "company_location", "company_size",
]

# Codes (always uppercase) vs free-text labels (smart title-case)
UPPER_COLS = {"salary_currency", "company_size"}

# Acronyms have a fixed canonical casing — preserve them inside titles.
ACRONYM_MAP = {
    "AI": "AI", "BI": "BI", "ML": "ML", "MLOPS": "MLOps",
    "ETL": "ETL", "NLP": "NLP", "AWS": "AWS", "DEVOPS": "DevOps",
}
# Connectives stay lowercase when not the first word.
LOWERCASE_WORDS = {"of", "and", "the", "in", "for", "to", "a", "an", "at", "on", "by"}


def smart_title(s):
    """Title-case a string while preserving known acronyms and connectives.

    Deterministic — works even for rare values where dirtify happened to
    corrupt the only/few clean instances (mode-based would pick the
    corrupted casing as canonical)."""
    if pd.isna(s):
        return s
    words = str(s).split()
    out = []
    for i, w in enumerate(words):
        if w.upper() in ACRONYM_MAP:
            out.append(ACRONYM_MAP[w.upper()])
        elif i > 0 and w.lower() in LOWERCASE_WORDS:
            out.append(w.lower())
        else:
            out.append(w.capitalize())
    return " ".join(out)


for col in CATEGORICAL_COLS:
    if col not in df.columns:
        continue
    s = df[col].astype("string").str.strip()
    if col in UPPER_COLS:
        s = s.str.upper()
    else:
        s = s.map(smart_title)
    df[col] = s

# Sanity: cardinality + spot-check no lingering upper/lower variants
for col in CATEGORICAL_COLS:
    if col in df.columns:
        print(f"  {col}: {df[col].nunique()} unique")
print("\nSpot check job_title (should be all Title Case):")
print(sorted(df["job_title"].dropna().unique().tolist())[:5])

  job_title: 125 unique
  job_category: 10 unique
  salary_currency: 11 unique
  employee_residence: 83 unique
  experience_level: 7 unique
  employment_type: 6 unique
  work_setting: 5 unique
  company_location: 70 unique
  company_size: 3 unique

Spot check job_title (should be all Title Case):
['AI Architect', 'AI Developer', 'AI Engineer', 'AI Programmer', 'AI Research Engineer']


In [5]:
# Step 3: Fix enum typos via mapping (case-insensitive lookup)
ENUM_MAPPING = {
    "experience_level": {
        "SNEIOR": "Senior", "SENIOR": "Senior",
        "MID LEVEL": "Mid-level", "MID-LEVEL": "Mid-level",
        "ENTRY": "Entry-level", "ENTRY-LEVEL": "Entry-level",
        "EXECUTIVE": "Executive",
    },
    "employment_type": {
        "FULLTIME": "Full-time", "FULL-TIME": "Full-time",
        "FRELANCE": "Freelance", "FREELANCE": "Freelance",
        "CONTRCT": "Contract", "CONTRACT": "Contract",
        "PART-TIME": "Part-time",
    },
    "work_setting": {
        "REMOT": "Remote", "REMOTE": "Remote",
        "ON-SITE": "In-person", "IN-PERSON": "In-person",
        "HYBRID": "Hybrid",
    },
    "company_size": {"L": "L", "M": "M", "S": "S"},
}

for col, mapping in ENUM_MAPPING.items():
    if col not in df.columns:
        continue
    upper_keys = {k.upper(): v for k, v in mapping.items()}
    df[col] = df[col].map(lambda v: upper_keys.get(str(v).strip().upper(), v) if pd.notna(v) else v)

for col in ENUM_MAPPING:
    if col in df.columns:
        print(f"{col}: {sorted(df[col].dropna().unique().tolist())}")

experience_level: ['Entry-level', 'Executive', 'Mid-level', 'Senior']
employment_type: ['Contract', 'Freelance', 'Full-time', 'Part-time']
work_setting: ['Hybrid', 'In-person', 'Remote']
company_size: ['L', 'M', 'S']


In [6]:
# Step 4: Parse `salary` back to numeric (drop $, commas, spaces)
def parse_salary(val):
    if pd.isna(val):
        return np.nan
    s = re.sub(r"[^0-9.\-]", "", str(val))
    return float(s) if s else np.nan

if "salary" in df.columns:
    before_dtype = df["salary"].dtype
    df["salary"] = df["salary"].map(parse_salary)
    print(f"salary dtype: {before_dtype} -> {df['salary'].dtype}")
    print(df["salary"].describe())

salary dtype: object -> float64
count      9495.000000
mean     149767.513639
std       63621.869282
min       14000.000000
25%      105200.000000
50%      143200.000000
75%      186600.000000
max      450000.000000
Name: salary, dtype: float64


In [7]:
# Step 5: Handle missing values
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing per column (before):")
print(missing)

# Strategy: drop rows missing key fields; impute mode for low-cardinality categoricals
KEY_COLS = ["work_year", "salary_in_usd", "job_title", "company_location"]
df = df.dropna(subset=[c for c in KEY_COLS if c in df.columns])

IMPUTE_MODE_COLS = ["salary_currency", "experience_level", "work_setting", "company_size", "job_category"]
for col in IMPUTE_MODE_COLS:
    if col in df.columns and df[col].isna().any():
        mode_val = df[col].mode(dropna=True).iloc[0]
        df[col] = df[col].fillna(mode_val)

print(f"\nMissing after: {int(df.isna().sum().sum())}")
print(f"Rows after dropna on key cols: {len(df)}")

Missing per column (before):
work_setting        401
company_size        391
experience_level    378
salary_currency     376
job_category        365
dtype: int64

Missing after: 0
Rows after dropna on key cols: 9495


In [8]:
# Step 6: Remove duplicates
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicates ({len(df)} rows remain)")

Removed 3976 duplicates (5519 rows remain)


In [9]:
# Step 7: Detect & handle outliers in salary_in_usd (IQR rule)
if "salary_in_usd" in df.columns:
    df["salary_in_usd"] = pd.to_numeric(df["salary_in_usd"], errors="coerce")
    q1, q3 = df["salary_in_usd"].quantile([0.25, 0.75])
    iqr = q3 - q1
    upper = q3 + 3 * iqr  # generous upper bound (3x instead of 1.5x)
    lower = max(0, q1 - 3 * iqr)
    n_out = ((df["salary_in_usd"] > upper) | (df["salary_in_usd"] < lower)).sum()
    print(f"Outlier bounds: [{lower:,.0f}, {upper:,.0f}] -> {n_out} outliers dropped")
    df = df[(df["salary_in_usd"] <= upper) & (df["salary_in_usd"] >= lower)].reset_index(drop=True)

Outlier bounds: [0, 448,575] -> 19 outliers dropped


In [10]:
# Step 8: Final dtype conversion + summary
DTYPES = {
    "work_year": "int16",
    "salary": "float64",
    "salary_in_usd": "int64",
}
for col, dt in DTYPES.items():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if dt.startswith("int"):
            df[col] = df[col].round().astype(dt)
        else:
            df[col] = df[col].astype(dt)

for col in CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].astype("category")

print(df.dtypes)
print(f"\nFinal shape: {df.shape}")
df.head()

work_year                int16
job_title             category
job_category          category
salary_currency       category
salary                 float64
salary_in_usd            int64
employee_residence    category
experience_level      category
employment_type       category
work_setting          category
company_location      category
company_size          category
dtype: object

Final shape: (5500, 12)


,work_year,job_title,job_category,salary_currency,salary,salary_in_usd,employee_residence,experience_level,employment_type,work_setting,company_location,company_size
0,2023,Data DevOps Engineer,Data Engineering,EUR,88000.0,95012,Germany,Mid-level,Full-time,Hybrid,Germany,L
1,2023,Data Architect,Data Architecture and Modeling,USD,186000.0,186000,United States,Senior,Full-time,In-person,United States,M
2,2023,Data Architect,Data Architecture and Modeling,USD,81800.0,81800,United States,Senior,Full-time,In-person,United States,M
3,2023,Data Scientist,Data Science and Research,USD,212000.0,212000,United States,Senior,Full-time,In-person,United States,M
4,2023,Data Scientist,Data Science and Research,USD,93300.0,93300,United States,Senior,Full-time,In-person,United States,M


In [11]:
# Step 9: Upload cleaned dataframe to MinIO (CSV + Parquet)
bucket = get_bucket_name()
csv_key = config["storage"]["cleaned_object_csv"]
parquet_key = config["storage"]["cleaned_object_parquet"]

upload_dataframe(df, object_key=csv_key, bucket=bucket, fmt="csv")
upload_dataframe(df, object_key=parquet_key, bucket=bucket, fmt="parquet")

print(f"\nDone. Bucket: {bucket}")
print(f"  - {csv_key}")
print(f"  - {parquet_key}")

Uploaded 5500 rows (662,234 bytes) -> s3://ds-salary/processed/jobs/cleaned_data.csv
Uploaded 5500 rows (60,026 bytes) -> s3://ds-salary/processed/jobs/cleaned_data.parquet

Done. Bucket: ds-salary
  - processed/jobs/cleaned_data.csv
  - processed/jobs/cleaned_data.parquet
